# 반도체 product failure 예측 분석

로지스틱 회귀 기반으로 반도체 공정/측정 데이터에서 failure를 예측하는 분류 모델을 구축합니다.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_generation import generate_synthetic_data
from src.preprocessing import load_data, preprocess
from src.model import split_data, train_logistic_regression
from src.evaluation import evaluate

## 1. 데이터 생성 및 확인

In [2]:
df = generate_synthetic_data()
df.head()

생성 완료: 5000행, 불량률 16.7%


,temperature,pressure,process_time,chemical_concentration,thickness,resistivity,dopant,failure
0,236.707694,2.963343,43.522975,3.540794,838.870803,22.143919,1.087762,0
1,142.885963,6.966349,42.963028,1.117792,566.914238,28.213500,0.980765,0
2,260.407418,7.901754,12.959007,3.090478,740.052201,23.128620,0.757356,0
3,215.263048,2.499725,3.162668,8.086393,906.572807,20.351918,0.083264,0
4,46.369657,1.394725,18.924172,4.985447,556.209859,41.596274,1.003711,0


In [3]:
df["failure"].value_counts(normalize=True).mul(100).round(1)

failure
0    83.3
1    16.7
Name: proportion, dtype: float64

## 2. 데이터 전처리

In [4]:
df = load_data("../data/synthetic_data.csv")
X, y, scaler = preprocess(df)
X.head()

데이터 로드: 5000행, 8열
결측치 처리: 991 -> 0
전처리 완료: X (5000, 7), y 불량 비율 16.7%


,temperature,pressure,process_time,chemical_concentration,thickness,resistivity,dopant
0,0.949195,-1.158680,0.765217,-0.537595,1.049759,-0.320589,0.616901
1,-0.299977,0.518415,0.732072,-1.377249,-0.489329,0.150293,0.509364
2,1.264740,0.910311,-1.043927,-0.693645,0.490513,-0.244195,0.284828
3,0.663674,-1.352916,-1.623793,1.037613,-0.004028,-0.459614,-0.392667
4,-1.585025,0.020985,-0.690837,-0.036972,-0.549908,1.188538,0.532426


## 3. 학습/테스트 분할 및 모델 학습

In [5]:
X_train, X_test, y_train, y_test = split_data(X, y)
model = train_logistic_regression(X_train, y_train)

데이터 분할: train 4000 / test 1000
  train 불량률: 16.8%, test 불량률: 16.7%
모델 학습 완료 (정규화 강도 C=1.0)


In [6]:
pd.Series(model.coef_[0], index=X.columns).sort_values(ascending=False)

thickness                 2.729340
temperature               2.110620
chemical_concentration    0.705925
dopant                    0.013919
process_time             -0.741288
resistivity              -0.748973
pressure                 -1.380634
dtype: float64

## 4. 모델 평가

In [7]:
import os
results = evaluate(model, X_test, y_test, plot_dir=os.path.abspath("../plots"))
results


모델 평가 결과 (로지스틱 회귀)
정확도(Accuracy)   : 0.9510 (95.1%)
정밀도(Precision)  : 0.8986 (89.9%)
재현율(Recall)     : 0.7964 (79.6%)
F1-score           : 0.8444 (84.4%)
AUC-ROC            : 0.9862



[저장] plots/confusion_matrix.png
[저장] plots/roc_curve.png


[저장] plots/precision_recall_curve.png
[저장] plots/feature_coefficients.png


{'accuracy': 0.951,
 'precision': 0.8986486486486487,
 'recall': 0.7964071856287425,
 'f1': 0.8444444444444444,
 'auc': 0.9862340145639094}

## 5. 교차 검증

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

cv_scores = cross_val_score(LogisticRegression(max_iter=1000, random_state=42), X, y, cv=5, scoring="accuracy")
print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

5-Fold CV Accuracy: 0.9462 (+/- 0.0058)
